In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
import os
import logging
from tensorflow.keras.models import Sequential, clone_model, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy, AUC
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import StandardScaler

In [2]:
# Set seed for reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [3]:
# Load the dataset
data_x = pd.read_csv('X_hml.csv')
data_y = pd.read_csv('y_hml.csv')

In [4]:
# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in data_x.columns:
    data_x = data_x.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in data_y.columns:
    data_y = data_y.drop(columns=['Unnamed: 0'])

In [5]:
# Feature selection
columns_to_keep = list(data_x.columns[:6]) + [
    'EMA_25_mkt', 'SMA_200_cma', 'EMA_26_rmw', 'EMA_26_hml',
    'EMA_25_smb', 'EMA_25_cma', 'RSI_9_cma', 'EMA_25_umd',
    'RSI_9_rmw', 'RSI_9_umd', 'RSI_9_smb', 'RSI_14_mkt'
]

X_train = data_x[columns_to_keep]
y_train = data_y['Target']

In [6]:
# Convert to numpy arrays
X = X_train.values
y = y_train.values

In [7]:
# Use StandardScaler to scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
# Convert back to DataFrame for EDA
X_scaled_df = pd.DataFrame(X_scaled, columns=columns_to_keep)

In [9]:
# Perform EDA on the final input matrix
X_scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
RET_cma,2129.0,-8.760802e-18,1.000235,-8.293726,-0.517883,-0.006036,0.525959,7.933323
RET_hml,2129.0,6.674897e-18,1.000235,-3.887676,-0.611578,-0.033145,0.587042,4.753445
RET_mkt,2129.0,2.336214e-17,1.000235,-11.363081,-0.448243,0.018535,0.479991,8.588009
RET_rmw,2129.0,1.168107e-17,1.000235,-5.501667,-0.596083,0.006557,0.610159,5.115630
RET_smb,2129.0,0.000000e+00,1.000235,-8.440059,-0.494293,0.030894,0.573561,6.091110
RET_umd,2129.0,-2.085905e-18,1.000235,-13.256709,-0.516088,-0.003167,0.519158,7.499463
EMA_25_mkt,2129.0,1.174782e-15,1.000235,-2.292113,-0.798175,0.276070,0.832795,1.495881
SMA_200_cma,2129.0,6.407901e-16,1.000235,-1.292052,-0.963404,-0.264407,0.801828,2.616652
EMA_26_rmw,2129.0,-1.922370e-15,1.000235,-2.421149,-0.617095,-0.129423,0.472417,3.368184
EMA_26_hml,2129.0,4.939423e-16,1.000235,-1.806230,-0.665746,-0.009678,0.605620,2.167290


In [10]:
# Define custom class weight function
def cwts(dfs):
    c0, c1 = np.bincount(dfs)
    w0 = (1 / c0) * (len(dfs)) / 2
    w1 = (1 / c1) * (len(dfs)) / 2
    return {0: w0, 1: w1}

In [11]:
# Define sliding window parameters
window_size = 600
test_gap = 180
step_size = 1

In [12]:
# Determine the start point to process only the last 500 windows
start_point = len(X) - (500 * step_size) - window_size - test_gap

In [13]:
# Lists to store results
all_test_predictions = []
all_actual_test_values = []
all_prediction_probabilities = []
all_predicted_classes = []

In [14]:
# Define model creation function
def create_model(hu=20, lookback=10, features=1):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=hu, input_shape=(lookback, features), activation='relu', return_sequences=False, name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))
    model.add(Dense(units=1, activation='sigmoid', name='Output'))
    opt = Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0)
    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    return model

In [ ]:
# Initialize and save the initial model
initial_model = create_model(hu=20, lookback=10, features=X.shape[1])
initial_model.compile(optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
initial_model.save('initial_model_MKT_Sigmoid&Tanh.h5')

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [ ]:
# Save the weights of the initial model
initial_model.save_weights('current_model_MKT_Sigmoid&Tanh.weights.h5')

In [17]:
# EarlyStopping callback
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

In [ ]:
# Checkpoint callback
checkpoint = ModelCheckpoint(filepath='model_checkpoint_MKT_Sigmoid&Tanh.weights.h5', save_best_only=True, save_weights_only=True, monitor='loss', mode='min')

In [ ]:
# Train and evaluate the model with sliding window
for start in range(start_point, len(X) - window_size - test_gap, step_size):
    try:
        end = start + window_size
        X_train_window = X[start:end]
        y_train_window = y[start:end]
        
        # Define test indices
        test_start = end + test_gap
        test_end = test_start + 1
        if test_end > len(X):
            break
        X_test_window = X[test_start:test_end]
        y_test_window = y[test_start:test_end]

        # Sequence length for training
        seqlen = 10
        # Number of features
        numfeat = X_train_window.shape[1]

        # Generate sequence input for the training dataset
        train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=32)
        
        # Calculate custom class weights using y_train_window
        class_weight = cwts(y_train_window)

        # Load and clone the initial model for the first iteration, then the previously trained model for subsequent iterations
        model = create_model(hu=20, lookback=10, features=numfeat)
        model.load_weights('current_model_MKT_Sigmoid&Tanh.weights.h5')

        model_clone = clone_model(model)
        model_clone.set_weights(model.get_weights())

        model_transfer = Sequential(model_clone.layers[:-1])
        model_transfer.add(Dense(1, activation='sigmoid'))

        # Freeze the cloned layers initially
        for layer in model_transfer.layers[:-1]:
            layer.trainable = False

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=Adam(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=4, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

        # Unfreeze the cloned layers for further training
        for layer in model_transfer.layers[:-1]:
            layer.trainable = True

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=Adam(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=50, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

        # Save the best model during training for the current window
        model_transfer.save('current_model_MKT_Sigmoid&Tanh.h5')
        model_transfer.save_weights('current_model_MKT_Sigmoid&Tanh.h5')

        # Predict on the single test sample directly
        test_input = np.expand_dims(X_test_window, axis=0)
        prediction = model_transfer.predict(test_input)
        prediction_probability = prediction[0][0]
        predicted_class = int(prediction_probability > 0.5)
        print(f'Test Prediction: {prediction_probability:.2f} (Class: {predicted_class})')

        # Store test predictions, actual values, and predicted classes
        all_test_predictions.append(prediction_probability)
        all_actual_test_values.append(y_test_window[0])
        all_prediction_probabilities.append(prediction_probability)
        all_predicted_classes.append(predicted_class)

    except Exception as e:
        print(f'Error at window {start} to {end}: {e}')
        break


Epoch 1/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4612 - auc_1: 0.4080 - loss: 11.7944    
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4829 - auc_1: 0.4171 - loss: 10.0139    
Epoch 3/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4490 - auc_1: 0.3932 - loss: 10.0646     
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4438 - auc_1: 0.3983 - loss: 10.2848     
Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3675 - auc_2: 0.3240 - loss: 10.5723    
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4414 - auc_2: 0.3511 - loss: 6.8375    
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4223 - auc_2: 0.3418 - loss: 4.1416     
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6173 - auc_2: 0.5133 - loss: 1.3174   
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5433 - auc_2: 0.4454 - loss: 1.0899     
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5878 - auc_2: 0.4080 - loss: 0.9405     
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5552 - auc_2: 0.3981 - loss: 0.8611     
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5808

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
Test Prediction: 0.85 (Class: 1)
Epoch 1/4


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3942 - auc_1: 0.3915 - loss: 0.7124    
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4361 - auc_1: 0.3928 - loss: 0.7059     
Epoch 3/4
 1/19 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.1562 - auc_1: 0.0000e+00 - loss: 0.8612

... (2,349 outputs left out)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6358 - auc_1: 0.1647 - loss: 0.6667        
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8159 - auc_1: 0.1402 - loss: 0.6645     
Epoch 3/4
 1/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - auc_1: 0.0000e+00 - loss: 0.6457

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8159 - auc_1: 0.3250 - loss: 0.6646     
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8159 - auc_1: 0.3250 - loss: 0.6649     
Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8159 - auc_2: 0.2901 - loss: 0.6642    
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6620     
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6621     
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6624     
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6627   
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6630     
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 - auc_2: 0.3250 - loss: 0.6633     
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8159 -

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step


In [ ]:
# Save the results to files for further analysis
results = {
    "test_predictions": all_test_predictions,
    "actual_test_values": all_actual_test_values,
    "prediction_probabilities": all_prediction_probabilities,
    "predicted_classes": all_predicted_classes
}

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('TF_results_MKT_Sigmoid&Tanh.csv', index=False)

In [ ]:
# Generate and print the combined classification report
combined_classification_report = classification_report(all_actual_test_values, all_predicted_classes, output_dict=True)
print("Combined Classification Report:")
print(combined_classification_report)

In [ ]:
# Calculate and print the combined confusion matrix
combined_cm = confusion_matrix(all_actual_test_values, all_predicted_classes)
print(f'Combined Confusion Matrix:\n{combined_cm}')

In [ ]:
# Save the final model
model_transfer.save('TF_model_MKT_Sigmoid&Tanh.h5')